<a href="https://colab.research.google.com/github/NidalNaaz/Int.-Msc-CS-AI-DS/blob/main/MachineLearning_LabCycle1Q6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Create a small user-item ratings dataset
data = {
    'User': ['U1','U1','U1','U2','U2','U2','U3','U3','U4','U4','U5','U5'],
    'Item': ['M1','M2','M3','M1','M2','M4','M2','M3','M1','M5','M4','M5'],
    'Rating': [5, 3, 4, 4, 2, 5, 2, 5, 5, 3, 4, 2]
}

df = pd.DataFrame(data)
print("User-Item Ratings Dataset:\n")
print(df)


User-Item Ratings Dataset:

   User Item  Rating
0    U1   M1       5
1    U1   M2       3
2    U1   M3       4
3    U2   M1       4
4    U2   M2       2
5    U2   M4       5
6    U3   M2       2
7    U3   M3       5
8    U4   M1       5
9    U4   M5       3
10   U5   M4       4
11   U5   M5       2


In [ ]:
# Pivot the dataframe to create a user-item matrix
user_item_matrix = df.pivot(index='User', columns='Item', values='Rating')

# Fill missing ratings with 0 (or NaN depending on your approach)
user_item_matrix_filled = user_item_matrix.fillna(0)

print("User-Item Matrix:\n")
print(user_item_matrix_filled)


User-Item Matrix:

Item   M1   M2   M3   M4   M5
User                         
U1    5.0  3.0  4.0  0.0  0.0
U2    4.0  2.0  0.0  5.0  0.0
U3    0.0  2.0  5.0  0.0  0.0
U4    5.0  0.0  0.0  0.0  3.0
U5    0.0  0.0  0.0  4.0  2.0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute cosine similarity between users
user_sim = cosine_similarity(user_item_matrix_filled)

# Convert to DataFrame for readability
user_sim_df = pd.DataFrame(user_sim, index=user_item_matrix_filled.index, columns=user_item_matrix_filled.index)

print("User Similarity Matrix (Cosine Similarity):\n")
print(user_sim_df)


User Similarity Matrix (Cosine Similarity):

User        U1        U2        U3        U4        U5
User                                                  
U1    1.000000  0.548128  0.682793  0.606339  0.000000
U2    0.548128  1.000000  0.110727  0.511310  0.666667
U3    0.682793  0.110727  1.000000  0.000000  0.000000
U4    0.606339  0.511310  0.000000  1.000000  0.230089
U5    0.000000  0.666667  0.000000  0.230089  1.000000


In [ ]:
def predict_ratings(user_item_matrix, user_sim):
    # Convert to numpy array for calculations
    ratings = user_item_matrix.values
    sim = user_sim.values

    # Weighted sum formula
    pred = sim.dot(ratings) / np.array([np.abs(sim).sum(axis=1)]).T
    pred_df = pd.DataFrame(pred, index=user_item_matrix.index, columns=user_item_matrix.columns)
    return pred_df

# Predict ratings
predicted_ratings = predict_ratings(user_item_matrix_filled, user_sim_df)
print("Predicted Ratings:\n")
print(predicted_ratings)


Predicted Ratings:

Item        M1        M2        M3        M4        M5
User                                                  
U1    3.603549  1.925041  2.613072  0.965946  0.641117
U2    3.277314  1.362731  0.968034  2.702545  1.010727
U3    2.150450  2.380700  4.310613  0.308687  0.000000
U4    4.292188  1.210372  1.033061  1.480960  1.473835
U5    2.012443  0.702955  0.000000  3.866250  1.418352


In [ ]:
def recommend_items(pred_df, user, top_n=2):
    """
    Recommend top N items for a given user based on predicted ratings.
    """
    # Get user's predicted ratings
    user_ratings = pred_df.loc[user]

    # Exclude items that user has already rated in the original dataset
    already_rated = user_item_matrix.loc[user] > 0
    unrated_ratings = user_ratings[~already_rated]

    # If all predicted ratings are zero (small dataset), just take top N anyway
    if len(unrated_ratings) == 0:
        unrated_ratings = user_ratings

    # Sort by predicted rating descending and pick top N
    recommendations = unrated_ratings.sort_values(ascending=False).head(top_n)
    return recommendations

# Example: Recommend top 2 items for U3
recommendations_U3 = recommend_items(predicted_ratings, 'U3')
print("Top Recommendations for U3:\n")
print(recommendations_U3)



Top Recommendations for U3:

Item
M1    2.150450
M4    0.308687
Name: U3, dtype: float64
